# 04 · Delay inherited within the operating day

**RQ2.** How much of the delay observed in a given hour is inherited from earlier
legs of the same aircraft?

BTS reports `LateAircraftDelay` as a cause, which is a label rather than a
mechanism. Reconstructing the aircraft rotation turns it into a quantity: the
previous leg's delay minus the scheduled turnaround buffer is the delay the next leg
cannot avoid.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

Team01-FlightDelay-Project v0.2.0 | mode=sample | years=2015-2024 | flights=flights_sample.parquet


In [2]:
from team01 import features, models, viz

panel = clean.completed(io.load_flights())
sample = models.take_sample(panel, 3_000_000)

legs = features.leg_sequence(sample)
cascade = features.cascade_summary(legs, sample)

print(f"rotations                    {cascade['rotations']:,}")
print(f"share of flights             {cascade['rotation_share_of_flights']:.1%}")
print(f"inherited share of delay     {cascade['inherited_share']:.1%}")

rotations                    23,055
share of flights             49.1%
inherited share of delay     32.4%


## 4.1 The arithmetic, on one aircraft

A rotation counts only when the aircraft continues from where it landed and the
scheduled ground time is plausible (15 minutes to 24 hours). The inherited minutes
are `previous delay − (scheduled turn − 25-minute minimum turn)`.

In [3]:
example = legs.loc[legs["is_rotation"]].head(6)
example[["tail_number", "origin", "prev_arr_delay", "scheduled_turn",
         "ground_time_actual", "inherited_min"]]

,tail_number,origin,prev_arr_delay,scheduled_turn,ground_time_actual,inherited_min
1,N1000,DFW,1.834014,69.0,67.165986,0.0
2,N1000,ATL,3.921912,91.0,87.078088,0.0
398,N1000,BOI,15.059569,45.0,29.940431,0.0
798,N1000,DTW,-14.118396,81.0,95.118396,0.0
2814,N1000,BOI,-1.266567,62.0,63.266567,0.0
2815,N1000,BTV,-5.866395,72.0,77.866395,0.0


## 4.2 Where it concentrates

Second-or-later legs late in the day inherit the most: the buffer is consumed by
mid-afternoon and every subsequent leg carries the delay forward.

In [4]:
cascade["hourly"][["dep_hour", "flights", "mean_inherited", "inherited_share"]].tail(10)

,dep_hour,flights,mean_inherited,inherited_share
8,13,2669,6.00,0.3160
9,14,2094,6.59,0.3303
10,15,1591,7.49,0.3615
11,16,1146,8.06,0.3558
12,17,859,8.84,0.4047
13,18,535,5.62,0.3128
14,19,316,10.31,0.4326
15,20,157,6.71,0.3098
16,21,64,6.95,0.3393
17,22,18,3.99,0.6720


In [5]:
viz.fig_cascade_hour(cascade["hourly"])

[viz] 08_cascade_hour.png


'C:\\Users\\lenovo\\Desktop\\Semester Project\\Team01-FlightDelay-Project\\figures\\08_cascade_hour.png'